# Analog Waveform Generation for Accordion.py

This notebook demonstrates how to generate multiple analog waveforms on MPIO channels 0-15 using the Accordion hardware wrapper. You'll learn how to:

- Create multiple waveform types (sine, square, sawtooth, triangle, etc.)
- Use threading for continuous background signal generation
- Dynamically adjust parameters in real-time from Jupyter Lab
- Manage multiple channels with different waveforms simultaneously

The probe on MPIO16 can be manually moved between pins to sample from different channels.

## Hardware Setup
- Output channels: MPIO0-MPIO15 (up to 16 waveforms)
- Monitor/Sample channel: MPIO16 (probe connected here)
- Voltage range: 0-5V (configurable)

## 1. Import Required Libraries and Set Up

In [ ]:
# Import required libraries
import sys
import time
import threading
import math
import random
from typing import Dict, List, Optional
from dataclasses import dataclass, field
from IPython.display import display, clear_output
import ipywidgets as widgets

# Import accordion module
import accordion as acc

# Connect to the hardware
acc.attach("py", "agentdemo.local")

# Configure MPIO00 through MPIO08 as outputs
for i in range(9):
    acc.configure_channelas_output(f"MPIO{i:02d}")

print("✓ Libraries imported, connected, and channels configured as outputs")

✓ Libraries imported and connected successfully


## 2. Waveform Functions

Define functions to generate various analog waveforms. Each function takes elapsed time and returns an instantaneous voltage value.

In [2]:
def sine_wave(t: float, frequency: float, amplitude: float, offset: float) -> float:
    """Sine wave: A*sin(2*pi*f*t) + offset"""
    return amplitude * math.sin(2 * math.pi * frequency * t) + offset

def square_wave(t: float, frequency: float, amplitude: float, offset: float) -> float:
    """Square wave: alternates between +amplitude and -amplitude"""
    period = 1.0 / frequency if frequency > 0 else 1.0
    phase = (t % period) / period
    return amplitude if phase < 0.5 else -amplitude + offset

def sawtooth_wave(t: float, frequency: float, amplitude: float, offset: float) -> float:
    """Sawtooth wave: linearly ramps from -amplitude to +amplitude"""
    period = 1.0 / frequency if frequency > 0 else 1.0
    phase = (t % period) / period
    return 2 * amplitude * (phase - 0.5) + offset

def triangle_wave(t: float, frequency: float, amplitude: float, offset: float) -> float:
    """Triangle wave: linear ramps up and down"""
    period = 1.0 / frequency if frequency > 0 else 1.0
    phase = (t % period) / period
    if phase < 0.25:
        return 4 * amplitude * phase + offset
    elif phase < 0.75:
        return 2 * amplitude * (0.5 - (phase - 0.25)) + offset
    else:
        return -4 * amplitude * (1 - phase) + offset

def ramp_wave(t: float, frequency: float, amplitude: float, offset: float) -> float:
    """Ramp wave: continuously decreasing sawtooth"""
    period = 1.0 / frequency if frequency > 0 else 1.0
    phase = (t % period) / period
    return 2 * amplitude * phase - amplitude + offset

def pulse_wave(t: float, frequency: float, amplitude: float, offset: float, duty_cycle: float = 0.5) -> float:
    """Pulse wave: adjustable duty cycle square wave"""
    period = 1.0 / frequency if frequency > 0 else 1.0
    phase = (t % period) / period
    return amplitude if phase < duty_cycle else -amplitude + offset

def chirp_wave(t: float, frequency: float, amplitude: float, offset: float, 
               freq_start: float = 1, freq_end: float = 100) -> float:
    """Chirp (frequency sweep) wave"""
    period = 1.0 / frequency if frequency > 0 else 1.0
    phase = (t % period) / period
    f = freq_start + (freq_end - freq_start) * phase
    instantaneous_phase = 2 * math.pi * (freq_start * phase + 
                                         (freq_end - freq_start) * phase**2 / 2)
    return amplitude * math.sin(instantaneous_phase) + offset

def noise_wave(t: float, frequency: float, amplitude: float, offset: float) -> float:
    """White noise: random values"""
    return random.uniform(-amplitude, amplitude) + offset

# Dictionary to look up waveforms
WAVEFORMS = {
    'sine': sine_wave,
    'square': square_wave,
    'sawtooth': sawtooth_wave,
    'triangle': triangle_wave,
    'ramp': ramp_wave,
    'pulse': pulse_wave,
    'chirp': chirp_wave,
    'noise': noise_wave,
}

print(f"✓ Defined {len(WAVEFORMS)} waveform types: {list(WAVEFORMS.keys())}")

✓ Defined 8 waveform types: ['sine', 'square', 'sawtooth', 'triangle', 'ramp', 'pulse', 'chirp', 'noise']


## 3. Channel Configuration and Waveform Generator Class

Create a data structure for channel configuration and the main generator class that manages threading and waveform output.

In [3]:
@dataclass
class ChannelConfig:
    """Configuration for a single waveform channel"""
    name: str
    waveform_type: str = 'sine'
    frequency: float = 10.0  # Hz
    amplitude: float = 2.5  # Volts
    offset: float = 2.5  # Volts
    enabled: bool = True
    vmin: float = 0.0
    vmax: float = 5.0
    extra_params: Dict = field(default_factory=dict)


class AnalogWaveformGenerator:
    """Generate multiple analog waveforms on MPIO channels with threading"""
    
    def __init__(self, update_rate: float = 1000, voltage_range: tuple = (0.0, 5.0)):
        """
        Args:
            update_rate: Updates per second (Hz)
            voltage_range: (min_voltage, max_voltage) for clamping
        """
        self.update_rate = update_rate
        self.vmin, self.vmax = voltage_range
        self.update_interval = 1.0 / update_rate
        
        self.channels: Dict[str, ChannelConfig] = {}
        self.running = False
        self.thread: Optional[threading.Thread] = None
        self.start_time = 0.0
        self._lock = threading.Lock()
    
    def add_channel(self, channel_name: str, waveform_type: str = 'sine', 
                   frequency: float = 10.0, amplitude: float = 2.5, 
                   offset: float = 2.5, **kwargs) -> None:
        """Add a waveform channel"""
        if waveform_type not in WAVEFORMS:
            print(f"⚠ Unknown waveform: {waveform_type}")
            waveform_type = 'sine'
        
        with self._lock:
            self.channels[channel_name] = ChannelConfig(
                name=channel_name,
                waveform_type=waveform_type,
                frequency=frequency,
                amplitude=amplitude,
                offset=offset,
                vmin=self.vmin,
                vmax=self.vmax,
                extra_params=kwargs
            )
    
    def _clamp_voltage(self, voltage: float) -> float:
        """Clamp voltage to safe range"""
        return max(self.vmin, min(self.vmax, voltage))
    
    def _generate_value(self, channel: ChannelConfig, elapsed_time: float) -> float:
        """Generate the current value for a channel"""
        if not channel.enabled:
            return channel.offset
        
        waveform_func = WAVEFORMS.get(channel.waveform_type, sine_wave)
        
        # Handle waveforms with extra parameters
        if channel.waveform_type == 'pulse':
            duty_cycle = channel.extra_params.get('duty_cycle', 0.5)
            value = waveform_func(elapsed_time, channel.frequency, 
                                 channel.amplitude, channel.offset, duty_cycle)
        elif channel.waveform_type == 'chirp':
            freq_start = channel.extra_params.get('freq_start', 1)
            freq_end = channel.extra_params.get('freq_end', 100)
            value = waveform_func(elapsed_time, channel.frequency,
                                 channel.amplitude, channel.offset,
                                 freq_start, freq_end)
        else:
            value = waveform_func(elapsed_time, channel.frequency,
                                 channel.amplitude, channel.offset)
        
        return self._clamp_voltage(value)
    
    def _generator_thread(self) -> None:
        """Main thread function for generating waveforms"""
        while self.running:
            with self._lock:
                elapsed_time = time.time() - self.start_time
                
                # Generate values for all enabled channels
                channel_names = []
                values = []
                
                for channel_name, channel in self.channels.items():
                    value = self._generate_value(channel, elapsed_time)
                    channel_names.append(channel_name)
                    values.append(f"{value:.4f}")
                
                # Set all values at once
                if channel_names and values:
                    try:
                        print(channel_names, values)
                        acc.set_values(channel_names, values)
                    except Exception as e:
                        print(f"Error: {e}")
            
            time.sleep(self.update_interval)
    
    def start(self) -> None:
        """Start generating waveforms"""
        if self.running:
            print("⚠ Generator already running")
            return
        
        if not self.channels:
            print("⚠ No channels configured")
            return
        
        self.running = True
        self.start_time = time.time()
        self.thread = threading.Thread(target=self._generator_thread, daemon=True)
        self.thread.start()
        print(f"✓ Started generating {len(self.channels)} waveforms")
    
    def stop(self) -> None:
        """Stop generating waveforms"""
        if not self.running:
            return
        
        self.running = False
        if self.thread:
            self.thread.join(timeout=2.0)
        print("✓ Stopped")
    
    def is_running(self) -> bool:
        """Check if generator is running"""
        return self.running
    
    def set_frequency(self, channel_name: str, frequency: float) -> None:
        """Change frequency of a channel"""
        with self._lock:
            if channel_name in self.channels:
                self.channels[channel_name].frequency = max(0.01, frequency)
    
    def set_amplitude(self, channel_name: str, amplitude: float) -> None:
        """Change amplitude of a channel"""
        with self._lock:
            if channel_name in self.channels:
                self.channels[channel_name].amplitude = amplitude
    
    def set_offset(self, channel_name: str, offset: float) -> None:
        """Change offset of a channel"""
        with self._lock:
            if channel_name in self.channels:
                self.channels[channel_name].offset = offset
    
    def set_waveform_type(self, channel_name: str, waveform_type: str) -> None:
        """Change waveform type of a channel"""
        with self._lock:
            if channel_name in self.channels:
                if waveform_type in WAVEFORMS:
                    self.channels[channel_name].waveform_type = waveform_type
    
    def enable_channel(self, channel_name: str, enabled: bool = True) -> None:
        """Enable or disable a channel"""
        with self._lock:
            if channel_name in self.channels:
                self.channels[channel_name].enabled = enabled
    
    def get_status(self) -> Dict:
        """Get status of all channels"""
        with self._lock:
            status = {
                'running': self.running,
                'num_channels': len(self.channels),
                'channels': {}
            }
            for name, ch in self.channels.items():
                status['channels'][name] = {
                    'type': ch.waveform_type,
                    'freq': ch.frequency,
                    'amp': ch.amplitude,
                    'offset': ch.offset,
                    'enabled': ch.enabled
                }
            return status
    
    def print_status(self) -> None:
        """Print formatted status"""
        status = self.get_status()
        print(f"\n{'='*70}")
        print(f"Analog Waveform Generator - {'RUNNING ✓' if status['running'] else 'STOPPED'}")
        print(f"Channels: {status['num_channels']}")
        print(f"{'='*70}")
        print(f"{'Channel':<12} {'Type':<12} {'Freq (Hz)':<12} {'Amp (V)':<12} {'Offset (V)':<12}")
        print(f"{'-'*70}")
        for name, cfg in status['channels'].items():
            print(f"{name:<12} {cfg['type']:<12} {cfg['freq']:<12.2f} "
                  f"{cfg['amp']:<12.2f} {cfg['offset']:<12.2f}")
        print(f"{'='*70}\n")

print("✓ AnalogWaveformGenerator class defined")

✓ AnalogWaveformGenerator class defined


## 4. Example 1: Three Basic Waveforms

Start with a simple example using sine, square, and sawtooth waves on MPIO0-2.

In [4]:
# Create generator instance
gen = AnalogWaveformGenerator(update_rate=0.5)

# Add three channels with different waveforms
gen.add_channel('MPIO00', waveform_type='sine', frequency=0.2, amplitude=2.0, offset=2.5)
gen.add_channel('MPIO01', waveform_type='square', frequency=0.1, amplitude=2.0, offset=2.5)
gen.add_channel('MPIO02', waveform_type='sawtooth', frequency=0.05, amplitude=2.0, offset=2.5)

# Show configuration
gen.print_status()

# Start generating
gen.start()


Analog Waveform Generator - STOPPED
Channels: 3
Channel      Type         Freq (Hz)    Amp (V)      Offset (V)  
----------------------------------------------------------------------
MPIO00       sine         0.20         2.00         2.50        
MPIO01       square       0.10         2.00         2.50        
MPIO02       sawtooth     0.05         2.00         2.50        

['MPIO00', 'MPIO01', 'MPIO02'] ['2.5000', '2.0000', '0.5000']
✓ Started generating 3 waveforms
Error: GetChannelFromNetName: [MPIO00] no such net name

   at System.Reflection.MethodBaseInvoker.InvokeWithFewArgs(Object obj, BindingFlags invokeAttr, Binder binder, Object[] parameters, CultureInfo culture)
   at ProtocolServerImpl.Reflection.ReflectionStub.FunctionInvocation(ReflectionCall call)
   at ProtocolServerImpl.Reflection.ReflectionStub.Server_ReflectionCallReceived(Guid g, ReflectionCall call)
   at ProtocolServerImpl.Reflection.ReflectionStub.SendReceive(ReflectionCall c) in C:\dev\source\accordionpilot

## 5. Dynamically Change Parameters (Live Adjustment)

While the generator is running, you can adjust frequency, amplitude, and offset on any channel in real-time. Use the cells below to experiment.

In [ ]:
# Example: Change frequency while running
# Uncomment and run these cells multiple times

# Increase MPIO0 frequency to 20 Hz
gen.set_frequency('MPIO00', 20)
print("MPIO00 frequency changed to 20 Hz")

# Or change amplitude
# gen.set_amplitude('MPIO00', 3.5)
# print("MPIO00 amplitude changed to 3.5 V")

# Or change waveform type
# gen.set_waveform_type('MPIO00', 'triangle')
# print("MPIO00 changed to triangle wave")

# Or change offset
# gen.set_offset('MPIO00', 1.5)
# print("MPIO00 offset changed to 1.5 V")

gen.print_status()

## 6. Example 2: All Waveform Types (Run after stopping previous example)

This demonstrates all 8 waveform types on channels MPIO0-7.

In [ ]:
# Stop previous generator if running
if gen.is_running():
    gen.stop()
    time.sleep(0.5)

# Create new generator for all waveforms
gen2 = AnalogWaveformGenerator(update_rate=500)

# Add all waveform types
waveforms = ['sine', 'square', 'sawtooth', 'triangle', 'ramp', 'chirp', 'pulse', 'noise']
for i, waveform_type in enumerate(waveforms):
    gen2.add_channel(
        f'MPIO{i:02d}',
        waveform_type=waveform_type,
        frequency=5 + i,  # Different frequencies: 5, 6, 7, 8, 9, 10, 11, 12 Hz
        amplitude=1.8,
        offset=2.5
    )

gen2.print_status()
gen2.start()

## 7. Example 3: Full 16-Channel Configuration

Generate all 16 channels (MPIO0-15) with varied waveforms.

In [ ]:
# Stop previous generator
if gen2.is_running():
    gen2.stop()
    time.sleep(0.5)

# Create 16-channel generator
gen3 = AnalogWaveformGenerator(update_rate=500)

# Add 16 channels with rotating waveforms
waveforms = ['sine', 'square', 'sawtooth', 'triangle', 'ramp', 'chirp', 'pulse', 'noise']
for i in range(16):
    waveform_type = waveforms[i % len(waveforms)]
    gen3.add_channel(
        f'MPIO{i:02d}',
        waveform_type=waveform_type,
        frequency=2 + i * 0.5,  # Different frequencies from 2 to 9.5 Hz
        amplitude=1.5,
        offset=2.5
    )

print(f"Configured {len(gen3.channels)} channels")
gen3.print_status()
gen3.start()

## 8. Interactive Control Panel

Use the controls below to adjust parameters in real-time while the generator is running.

In [ ]:
# Select active generator (whichever is running)
active_gen = gen3 if gen3.is_running() else (gen2 if gen2.is_running() else gen)

# Create interactive controls
output = widgets.Output()

def on_frequency_change(change):
    channel = channel_select.value
    freq = frequency_slider.value
    active_gen.set_frequency(channel, freq)
    with output:
        clear_output(wait=True)
        print(f"✓ {channel} frequency: {freq} Hz")

def on_amplitude_change(change):
    channel = channel_select.value
    amp = amplitude_slider.value
    active_gen.set_amplitude(channel, amp)
    with output:
        clear_output(wait=True)
        print(f"✓ {channel} amplitude: {amp} V")

def on_waveform_change(change):
    channel = channel_select.value
    waveform = waveform_select.value
    active_gen.set_waveform_type(channel, waveform)
    with output:
        clear_output(wait=True)
        print(f"✓ {channel} waveform: {waveform}")

# Channel selector
channel_select = widgets.Dropdown(
    options=list(active_gen.channels.keys()),
    description='Channel:'
)

# Frequency slider
frequency_slider = widgets.FloatSlider(
    value=10,
    min=0.1,
    max=100,
    step=1,
    description='Frequency (Hz):'
)
frequency_slider.observe(on_frequency_change, names='value')

# Amplitude slider
amplitude_slider = widgets.FloatSlider(
    value=1.5,
    min=0.1,
    max=5.0,
    step=0.1,
    description='Amplitude (V):'
)
amplitude_slider.observe(on_amplitude_change, names='value')

# Waveform selector
waveform_select = widgets.Dropdown(
    options=list(WAVEFORMS.keys()),
    description='Waveform:'
)
waveform_select.observe(on_waveform_change, names='value')

# Display controls
display(widgets.VBox([
    channel_select,
    frequency_slider,
    amplitude_slider,
    waveform_select,
    output
]))

print("✓ Interactive controls ready")

## 9. Stop All Generators

Run this cell to stop all waveform generation.

In [ ]:
# Stop all generators
for g in [gen, gen2, gen3]:
    if g.is_running():
        g.stop()
        print(f"✓ Stopped generator with {len(g.channels)} channels")

print("\n✓ All generators stopped")

## 10. Advanced Topics

### Creating Custom Quick-Setup Presets

```python
def setup_test_configuration():
    """Quick setup for testing"""
    gen = AnalogWaveformGenerator(update_rate=1000)
    
    # Add channels with specific configuration
    configs = [
        ('MPIO00', 'sine', 10, 2.0, 2.5),
        ('MPIO01', 'square', 5, 2.0, 2.5),
        ('MPIO02', 'triangle', 3, 2.0, 2.5),
        ('MPIO03', 'ramp', 2, 2.0, 2.5),
    ]
    
    for ch_name, waveform, freq, amp, offset in configs:
        gen.add_channel(ch_name, waveform, freq, amp, offset)
    
    return gen

# Usage:
# gen_test = setup_test_configuration()
# gen_test.start()
```

### Monitoring Performance

The generator uses threading, so it runs independently. You can check status at any time:

```python
status = gen.get_status()
print(f"Running: {status['running']}")
print(f"Channels: {status['num_channels']}")
```

### Notes

- **Update Rate**: Higher rates (1000+ Hz) provide smoother waveforms but may increase CPU load
- **Voltage Range**: Default is 0-5V. Adjust via `voltage_range` parameter
- **Threading**: The generator runs on a daemon thread, so it won't block notebook execution
- **Thread Safety**: All parameter changes are thread-safe using locks
- **Probe on MPIO16**: Manually move the probe between channels to sample different signals